# Dual-retriever offline evaluation — cases

Evaluates **both** case retrievers fairly, each on a gold set grounded in its own
corpus. No API / LLM calls.

| Retriever | Index | Model | Gold set | Coverage |
|---|---|---|---|---|
| Chat RAG cases | `faiss_index_cases` | bge-base-en-v1.5 (768-d) | `golden_queries_bge.jsonl` (built from indexed cases) | 100% |
| Case-search tool | `faiss_case_retrieval` | Qwen3-Embedding-0.6B (1024-d) | `golden_queries.jsonl` (your original) | 100% |

Pipeline: coverage audit -> Recall@1/5/10, MRR@10, nDCG@10 with bootstrap 95% CIs
-> per-query-type breakdown -> unanswerable gate (AUC + threshold sweep + honest
dev/test split) -> FETCH_K sensitivity sweep -> side-by-side summary.

**Caveat:** the two corpora and gold sets differ, so this measures each retriever
on its own corpus, not an identical-query head-to-head. The bge corpus has ~30%
cross-case chunk contamination (see `build_gold_cases.py`), which caps precision.

Run all cells top-to-bottom. Writes `eval_results_bge.json`, `eval_results_qwen.json`,
`eval_summary_dual.txt`.


In [1]:
import json, pickle, re, random, sqlite3, gc, os, sys
import numpy as np
import faiss, torch
from sentence_transformers import SentenceTransformer
sys.path.insert(0, os.path.abspath(".."))

BGE_DIR = "../faiss_index_cases"; BGE_GOLD = "golden_queries_bge.jsonl"
BGE_MODEL = "BAAI/bge-base-en-v1.5"
BGE_PREFIX = "Represent this sentence for searching relevant passages: "
QWEN_DIR = "../faiss_case_retrieval"; QWEN_GOLD = "golden_queries.jsonl"
FETCH_K = 200; TOP_K = 10; SEED = 42; N_BOOT = 2000
SWEEP = (100, 200, 400)
THRESH_GRID = np.round(np.arange(0.30, 0.901, 0.01), 2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
lines = []
def say(s=""):
    print(s, flush=True); lines.append(s)


In [2]:
def canon(s):
    s = re.sub(r"(?i)citation\s*name\s*:", " ", s or "")
    s = re.split(r"(?i)bookmark", s)[0]
    return re.sub(r"[^a-z0-9]", "", s.lower())

def to_cos_fn(index):
    is_l2 = index.metric_type == faiss.METRIC_L2
    return (lambda d: 1.0 - d / 2.0 if is_l2 else float(d)), ("L2" if is_l2 else "IP")

def search(Q, index, chunk_keys, to_cos, fetch_k, top_k):
    D, I = index.search(Q, fetch_k)
    out = []
    for drow, irow in zip(D, I):
        best = {}
        for d, i in zip(drow, irow):
            if i < 0: continue
            k = chunk_keys[int(i)]
            if not k: continue
            s = to_cos(float(d))
            if k not in best or s > best[k]: best[k] = s
        out.append(sorted(best.items(), key=lambda kv: -kv[1])[:top_k])
    return out

def records(rows, ranked_all, index_keys):
    res = []
    for row, ranked in zip(rows, ranked_all):
        keys = [k for k, _ in ranked]
        rec = {"id": row["id"], "query": row["query"], "query_type": row["query_type"],
               "answerable": row["answerable"],
               "top_score": float(ranked[0][1]) if ranked else -1.0,
               "top_cases": [{"citation_norm": k, "cos": round(s, 4)} for k, s in ranked[:5]]}
        if row["answerable"]:
            gold = {canon(c) for c in row["relevant_citations"]} - {""}
            rec["gold"] = sorted(gold); rec["gold_in_index"] = bool(gold & index_keys)
            rec["rank"] = next((i + 1 for i, k in enumerate(keys) if k in gold), None)
        res.append(rec)
    return res

def retrieval_metrics(recs):
    ranks = [r["rank"] for r in recs]
    def hit(K): return [1.0 if (r is not None and r <= K) else 0.0 for r in ranks]
    return {"Recall@1": hit(1), "Recall@5": hit(5), "Recall@10": hit(10),
            "MRR@10": [1.0 / r if (r is not None and r <= TOP_K) else 0.0 for r in ranks],
            "nDCG@10": [1.0 / np.log2(r + 1) if (r is not None and r <= TOP_K) else 0.0 for r in ranks]}

def bootstrap_ci(vals, rng):
    v = np.asarray(vals, float)
    if len(v) == 0: return 0.0, 0.0, 0.0
    idx = rng.integers(0, len(v), size=(N_BOOT, len(v))); means = v[idx].mean(axis=1)
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

def show_block(title, recs, rng):
    say(f"\n{title}  (n={len(recs)})")
    if not recs: say("  no queries"); return {}
    m = {}
    for name, vals in retrieval_metrics(recs).items():
        mu, lo, hi = bootstrap_ci(vals, rng); m[name] = mu
        say(f"  {name:10s} {mu:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
    return m

def auc(pos, neg):
    pos, neg = np.asarray(pos), np.asarray(neg)
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    return float((pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean())

def gate_stats(a, u, thr):
    tpr = float(np.mean(np.asarray(a) >= thr)) if len(a) else 0.0
    tnr = float(np.mean(np.asarray(u) < thr)) if len(u) else 0.0
    return (tpr + tnr) / 2, tpr, tnr

def load_bge():
    index = faiss.read_index(f"{BGE_DIR}/index.faiss")
    with open(f"{BGE_DIR}/index.pkl", "rb") as f: docstore, id_map = pickle.load(f)
    d = getattr(docstore, "_dict", docstore)
    chunk_keys = [canon((d[id_map[i]].metadata or {}).get("citation", "")) for i in range(index.ntotal)]
    model = SentenceTransformer(BGE_MODEL, device=DEVICE)
    def enc(qs):
        return model.encode([BGE_PREFIX + q for q in qs], normalize_embeddings=True,
                            batch_size=32, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    return index, chunk_keys, enc, model

def load_qwen():
    from qwen_embeddings import get_qwen_model
    info = json.loads(open(f"{QWEN_DIR}/run_info.json", encoding="utf-8").read())
    index = faiss.read_index(f"{QWEN_DIR}/index.faiss")
    con = sqlite3.connect(f"file:{QWEN_DIR}/metadata.sqlite3?mode=ro", uri=True)
    cit = {cid: canon(c) for cid, c in con.execute("SELECT case_id, citation FROM cases")}
    chunk_keys = [""] * index.ntotal
    for vid, cid in con.execute("SELECT vector_id, case_id FROM chunks"):
        if 0 <= vid < index.ntotal: chunk_keys[vid] = cit.get(cid, "")
    con.close()
    model = get_qwen_model(info["model_name"], device="cpu", max_seq_length=int(info.get("max_seq_length", 2048)))
    prompt = str(info.get("query_prompt", ""))
    def enc(qs):
        return model.encode([prompt + q for q in qs], normalize_embeddings=True,
                            batch_size=16, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    return index, chunk_keys, enc, model, info

def evaluate(tag, name, index, chunk_keys, enc, gold_file, model_desc):
    say(f"\n########## {name} ##########")
    rows = [json.loads(l) for l in open(gold_file, encoding="utf-8") if l.strip()]
    to_cos, metric = to_cos_fn(index); index_keys = set(chunk_keys) - {""}
    v0 = index.reconstruct(0)
    say("=== Setup ===")
    say(f"Vectors {index.ntotal}  dim {index.d}  metric {metric}  vec0 norm {np.linalg.norm(v0):.4f}")
    say(f"Distinct cases {len(index_keys)}  | model {model_desc}  device {DEVICE}")
    say(f"Gold {gold_file}  rows {len(rows)}  FETCH_K/TOP_K {FETCH_K}/{TOP_K}")
    gold_all = {canon(c) for r in rows if r["answerable"] for c in r["relevant_citations"]} - {""}
    found = gold_all & index_keys; cov = 100 * len(found) / max(len(gold_all), 1)
    say(f"Gold cases {len(gold_all)}  found {len(found)} ({cov:.1f}%)")
    if cov < 90: say("WARNING: gold coverage <90% - scores capped by missing cases.")
    Q = enc([r["query"] for r in rows])
    ranked = search(Q, index, chunk_keys, to_cos, FETCH_K, TOP_K)
    res = records(rows, ranked, index_keys)
    json.dump(res, open(f"eval_results_{tag}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    rng = np.random.default_rng(SEED)
    ans = [r for r in res if r["answerable"]]; un = [r for r in res if not r["answerable"]]
    ceil = [r for r in ans if r["gold_in_index"]]
    say("\n=== Retrieval: all answerable ==="); show_block("All answerable", ans, rng)
    say("\n=== Retrieval: ceiling (gold in index) ==="); cm = show_block("Gold in index", ceil, rng)
    say("\n=== By query type (gold in index) ===")
    for qt in sorted({r["query_type"] for r in ceil}):
        show_block(qt, [r for r in ceil if r["query_type"] == qt], rng)
    a = [r["top_score"] for r in ans]; u = [r["top_score"] for r in un]
    A = auc(a, u)
    say("\n=== Unanswerable gate ===")
    say(f"AUC {A:.3f}  (0.5 chance, 1.0 perfect)")
    say("threshold sweep (TPR=answerable accepted, TNR=unanswerable refused):")
    for t in (0.60, 0.65, 0.70, 0.72, 0.75, 0.80):
        bal, tpr, tnr = gate_stats(a, u, t)
        say(f"  thr {t:.2f}  TPR {tpr:.3f}  TNR {tnr:.3f}  balanced {bal:.3f}")
    grp = sorted({tuple(r["gold"]) for r in ans}); random.Random(SEED).shuffle(grp)
    devg = set(grp[:len(grp) // 2])
    ui = list(range(len(un))); random.Random(SEED + 1).shuffle(ui); devu = set(ui[:len(un) // 2])
    da = [r["top_score"] for r in ans if tuple(r["gold"]) in devg]
    ta = [r["top_score"] for r in ans if tuple(r["gold"]) not in devg]
    du = [un[i]["top_score"] for i in range(len(un)) if i in devu]
    tu = [un[i]["top_score"] for i in range(len(un)) if i not in devu]
    thr = float(max(THRESH_GRID, key=lambda t: gate_stats(da, du, t)[0]))
    bal, tpr, tnr = gate_stats(ta, tu, thr)
    say(f"dev-chosen thr {thr:.2f} -> test TPR {tpr:.3f} (n={len(ta)})  TNR {tnr:.3f} (n={len(tu)})  balanced {bal:.3f}")
    say(f"top-score p10/50/90 answerable   {np.percentile(a, [10, 50, 90]).round(3)}")
    say(f"top-score p10/50/90 unanswerable {np.percentile(u, [10, 50, 90]).round(3)}")
    say("\n=== FETCH_K sweep (ceiling Recall@10) ===")
    sweep = {}
    for fk in SWEEP:
        rk = search(Q, index, chunk_keys, to_cos, fk, TOP_K)
        rc = records(rows, rk, index_keys)
        cl = [r for r in rc if r["answerable"] and r["gold_in_index"]]
        ranks = [r["rank"] for r in cl]
        r10 = float(np.mean([1.0 if (x is not None and x <= 10) else 0.0 for x in ranks])) if ranks else 0.0
        sweep[fk] = round(r10, 3); say(f"  FETCH_K {fk:4d}  Recall@10 {r10:.3f}  (n={len(cl)})")
    return {"name": name, "model": model_desc, "vectors": index.ntotal, "cases": len(index_keys),
            "gold_rows": len(rows), "coverage": round(cov, 1), "ceiling_n": len(ceil),
            "R1": round(cm.get("Recall@1", 0), 3), "R5": round(cm.get("Recall@5", 0), 3),
            "R10": round(cm.get("Recall@10", 0), 3), "MRR": round(cm.get("MRR@10", 0), 3),
            "AUC": round(A, 3), "gate_thr": thr, "gate_tpr": round(tpr, 3), "gate_tnr": round(tnr, 3),
            "sweep": sweep}


In [3]:
index, chunk_keys, enc, model = load_bge()
SUM_BGE = evaluate("bge", "bge-base  |  faiss_index_cases  |  grounded gold",
                   index, chunk_keys, enc, BGE_GOLD, BGE_MODEL)
del index, chunk_keys, enc, model; gc.collect()



########## bge-base  |  faiss_index_cases  |  grounded gold ##########
=== Setup ===
Vectors 209168  dim 768  metric L2  vec0 norm 1.0000
Distinct cases 15627  | model BAAI/bge-base-en-v1.5  device cpu
Gold golden_queries_bge.jsonl  rows 480  FETCH_K/TOP_K 200/10
Gold cases 150  found 150 (100.0%)

=== Retrieval: all answerable ===

All answerable  (n=450)
  Recall@1   0.169   95% CI [0.133, 0.202]
  Recall@5   0.224   95% CI [0.184, 0.264]
  Recall@10  0.247   95% CI [0.207, 0.289]
  MRR@10     0.194   95% CI [0.160, 0.228]
  nDCG@10    0.206   95% CI [0.170, 0.243]

=== Retrieval: ceiling (gold in index) ===

Gold in index  (n=450)
  Recall@1   0.169   95% CI [0.136, 0.202]
  Recall@5   0.224   95% CI [0.187, 0.267]
  Recall@10  0.247   95% CI [0.209, 0.284]
  MRR@10     0.194   95% CI [0.160, 0.229]
  nDCG@10    0.206   95% CI [0.173, 0.241]

=== By query type (gold in index) ===

case_name  (n=150)
  Recall@1   0.067   95% CI [0.033, 0.107]
  Recall@5   0.093   95% CI [0.053, 0.14

In [4]:
index, chunk_keys, enc, model, info = load_qwen()
SUM_QWEN = evaluate("qwen", "Qwen3  |  faiss_case_retrieval  |  original gold",
                    index, chunk_keys, enc, QWEN_GOLD, info["model_name"])
del index, chunk_keys, enc; gc.collect()



########## Qwen3  |  faiss_case_retrieval  |  original gold ##########
=== Setup ===
Vectors 124561  dim 1024  metric L2  vec0 norm 1.0000
Distinct cases 43960  | model Qwen/Qwen3-Embedding-0.6B  device cpu
Gold golden_queries.jsonl  rows 439  FETCH_K/TOP_K 200/10
Gold cases 200  found 200 (100.0%)

=== Retrieval: all answerable ===

All answerable  (n=400)
  Recall@1   0.170   95% CI [0.135, 0.210]
  Recall@5   0.307   95% CI [0.263, 0.352]
  Recall@10  0.383   95% CI [0.338, 0.432]
  MRR@10     0.229   95% CI [0.195, 0.266]
  nDCG@10    0.265   95% CI [0.229, 0.302]

=== Retrieval: ceiling (gold in index) ===

Gold in index  (n=400)
  Recall@1   0.170   95% CI [0.135, 0.207]
  Recall@5   0.307   95% CI [0.263, 0.352]
  Recall@10  0.383   95% CI [0.333, 0.430]
  MRR@10     0.229   95% CI [0.192, 0.266]
  nDCG@10    0.265   95% CI [0.227, 0.303]

=== By query type (gold in index) ===

direct  (n=200)
  Recall@1   0.195   95% CI [0.140, 0.255]
  Recall@5   0.335   95% CI [0.270, 0.405]

In [5]:
say("\n########## SIDE-BY-SIDE (each retriever on its own grounded gold) ##########")
say(f"{'metric':24}{'bge/cases':>14}{'Qwen/case_retr':>17}")
for label, k in [("vectors", "vectors"), ("distinct cases", "cases"), ("gold rows", "gold_rows"),
                 ("gold coverage %", "coverage"), ("ceiling n", "ceiling_n"),
                 ("ceiling Recall@1", "R1"), ("ceiling Recall@5", "R5"), ("ceiling Recall@10", "R10"),
                 ("ceiling MRR@10", "MRR"), ("gate AUC", "AUC"), ("gate thr (dev)", "gate_thr"),
                 ("gate TPR (test)", "gate_tpr"), ("gate TNR (test)", "gate_tnr")]:
    say(f"{label:24}{str(SUM_BGE[k]):>14}{str(SUM_QWEN[k]):>17}")
say("\nNote: different corpora + gold sets, so each retriever is measured fairly on its")
say("own corpus, NOT an identical-query head-to-head. bge corpus has ~30% cross-case")
say("chunk contamination, which caps its precision.")
open("eval_summary_dual.txt", "w", encoding="utf-8").write("\n".join(lines) + "\n")
say("\nSaved: eval_results_bge.json, eval_results_qwen.json, eval_summary_dual.txt")



########## SIDE-BY-SIDE (each retriever on its own grounded gold) ##########
metric                       bge/cases   Qwen/case_retr
vectors                         209168           124561
distinct cases                   15627            43960
gold rows                          480              439
gold coverage %                  100.0            100.0
ceiling n                          450              400
ceiling Recall@1                 0.169             0.17
ceiling Recall@5                 0.224            0.308
ceiling Recall@10                0.247            0.382
ceiling MRR@10                   0.194            0.229
gate AUC                         0.773            0.569
gate thr (dev)                    0.69             0.67
gate TPR (test)                   0.56            0.605
gate TNR (test)                    0.8             0.55

Note: different corpora + gold sets, so each retriever is measured fairly on its
own corpus, NOT an identical-query head-to-head. bge cor